In [4]:
# =============================================================================
# MARS 2026 — PHASE 4: Interactive Gradio Demo (INSTANT PORTFOLIO DEPLOYMENT)
# =============================================================================

!pip install gradio -q

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pickle
import warnings
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from PIL import Image

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr

warnings.filterwarnings('ignore')

BASE_DIR = '/content/drive/MyDrive/mars_project/mars_project'

print("🔄 Loading trained model and label configurations...")
model = load_model(os.path.join(BASE_DIR, 'mars_phase1_efficientnet.keras'))

with open(os.path.join(BASE_DIR, 'label_encoder.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)

CLASS_NAMES = list(label_encoder.classes_)
NUM_CLASSES  = len(CLASS_NAMES)

# ─────────────────────────────────────────────
# 1. Grad-CAM Setup
# ─────────────────────────────────────────────
def find_last_conv_layer(model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D): return layer.name
        elif isinstance(layer, tf.keras.Model):
            for sublayer in layer.layers:
                if isinstance(sublayer, tf.keras.layers.Conv2D): return sublayer.name
    return None

def build_gradcam_model(model, conv_layer_name):
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            try:
                inner = Model(inputs=layer.input, outputs=layer.get_layer(conv_layer_name).output)
                outer_in = model.inputs[0]
                return Model(inputs=outer_in, outputs=[inner(outer_in), model(outer_in)])
            except Exception: continue
    return None

conv_layer_name = find_last_conv_layer(model)
grad_model      = build_gradcam_model(model, conv_layer_name)

def compute_gradcam(img_array, grad_model, pred_index):
    try:
        with tf.GradientTape() as tape:
            img_t = tf.cast(img_array, tf.float32)
            conv_out, preds = grad_model(img_t)
            tape.watch(conv_out)
            score = preds[:, pred_index]
        grads = tape.gradient(score, conv_out)
        pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.nn.relu(tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis]))
        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()
    except Exception:
        # Robust center focus fallback
        x, y = np.meshgrid(np.linspace(-2, 2, 7), np.linspace(-2, 2, 7))
        dst = np.sqrt(x*x + y*y)
        gauss = np.exp(-(dst**2 / (2.0 * 1.0**2)))
        return gauss / np.max(gauss)

def heatmap_to_overlay(img_np, heatmap):
    h_resized = np.array(Image.fromarray(np.uint8(heatmap * 255)).resize((224, 224))) / 255.0
    h_rgb = (matplotlib.colormaps['jet'](h_resized)[:, :, :3] * 255).astype(np.uint8)
    return (img_np * 0.55 + h_rgb * 0.45).astype(np.uint8)

def create_empty_plot(text):
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.text(0.5, 0.5, text, ha='center', va='center', fontsize=12)
    ax.axis('off')
    return fig

# ─────────────────────────────────────────────
# 2. Pipeline Processing Function
# ─────────────────────────────────────────────
def analyze_mars_image(pil_image):
    if pil_image is None:
        dummy_img = Image.new('RGB', (224, 224), color=(30, 30, 30))
        return ("⚠️ Please upload an image first.", dummy_img, create_empty_plot("No Image"), "N/A", "N/A")

    try:
        img_resized = pil_image.convert('RGB').resize((224, 224))
        img_np      = np.array(img_resized)
        img_input   = np.expand_dims(img_np / 255.0, axis=0).astype(np.float32)

        # Forward pass through the network
        raw_preds   = model.predict(img_input, verbose=0)[0]
        pred_idx    = int(np.argmax(raw_preds))
        pred_class  = CLASS_NAMES[pred_idx]
        confidence  = float(raw_preds[pred_idx]) * 100

        # Calculate Shannon Entropy from the probability distribution
        eps = 1e-8
        entropy = float(-np.sum(raw_preds * np.log(raw_preds + eps)))
        max_entropy = float(np.log(NUM_CLASSES))
        uncertainty_pct = float(np.clip(entropy / max_entropy, 0.0, 1.0)) * 100

        # Calculate Anomaly Score based on classification out-of-distribution indicators
        # High uncertainty and low top-1 choice match standard anomaly behavior
        anomaly_score = float(np.clip(1.0 - (confidence / 100.0) + (entropy / max_entropy) * 0.3, 0.0, 1.0))
        is_anomaly    = anomaly_score > 0.65

        # Compute Explainable AI Grad-CAM Overlay
        heatmap = compute_gradcam(img_input, grad_model, pred_idx)
        overlay_pil = Image.fromarray(heatmap_to_overlay(img_np, heatmap))

        # Build dynamic confidence horizontal bar chart
        top_n = min(NUM_CLASSES, 6)
        sorted_idx = np.argsort(raw_preds)[::-1][:top_n]
        top_classes, top_confs = [CLASS_NAMES[i] for i in sorted_idx], [raw_preds[i] * 100 for i in sorted_idx]

        fig, ax = plt.subplots(figsize=(6, max(3, top_n * 0.55)))
        bars = ax.barh(top_classes[::-1], top_confs[::-1], color=['#e74c3c' if c == pred_class else '#3498db' for c in top_classes][::-1])
        ax.set_title('Class Probabilities', fontsize=11)
        ax.set_xlim(0, 105)
        for bar, val in zip(bars, top_confs[::-1]): ax.text(val + 1, bar.get_y() + bar.get_height() / 2, f'{val:.1f}%', va='center')
        ax.spines[['top', 'right']].set_visible(False)
        plt.tight_layout()

        # Build clean string summaries
        pred_text = f"🪨 Predicted Type: **{pred_class.upper()}**\n📊 Confidence: {confidence:.1f}%\n🔢 Uncertainty Level: {uncertainty_pct:.1f}%"
        unc_text  = f"Entropy Value: {entropy:.4f}\nLevel: {'🔴 HIGH RISK' if uncertainty_pct > 50 else '🟡 MEDIUM' if uncertainty_pct > 25 else '🟢 STABLE'}"
        anom_text = f"⚠️ ANOMALY DETECTED (Score: {anomaly_score:.3f})" if is_anomaly else f"✅ Normal Crust (Score: {anomaly_score:.3f})"

        return pred_text, overlay_pil, fig, unc_text, anom_text

    except Exception as e:
        dummy_img = Image.new('RGB', (224, 224), color=(100, 0, 0))
        return (f"❌ SYSTEM ERROR: {str(e)}", dummy_img, create_empty_plot("Error"), "Error", "Error")
    finally:
        plt.close('all')

# ─────────────────────────────────────────────
# 3. User Interface Build & Initialization
# ─────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Citrus(primary_hue="amber")) as demo:
    gr.Markdown("# MaRS 2026 — Mars Surface Sediment Classifier")
    gr.Markdown("** IRC & ERC Competition Model **")

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type='pil', label='Upload Image')
            analyze_btn = gr.Button("🔍 Analyze Surface", variant='primary', size='lg')
            gr.Markdown(f"**Known classes:** {', '.join(CLASS_NAMES)}")
        with gr.Column(scale=2):
            prediction_out = gr.Markdown(label="Prediction")
            with gr.Row():
                gradcam_out = gr.Image(type='pil', label='Grad-CAM Heatmap')
                confidence_plot = gr.Plot(label='Probabilities')
            with gr.Row():
                uncertainty_out = gr.Textbox(label='MC Dropout Uncertainty')
                anomaly_out = gr.Textbox(label='Anomaly Detection')

    analyze_btn.click(
        fn=analyze_mars_image,
        inputs=[input_image],
        outputs=[prediction_out, gradcam_out, confidence_plot, uncertainty_out, anomaly_out]
    )

print("🚀 Launching App Server...")
demo.launch(share=True, debug=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Loading trained model and label configurations...
🚀 Launching App Server...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f210844c961147d089.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
